In [ ]:
import torch
import pandas as pd
import numpy as np

from datasets import load_dataset

from transformers import AutoTokenizer,AutoModelForQuestionAnswering,TrainingArguments,Trainer,DefaultDataCollator


from sklearn.metrics import accuracy_score, f1_score

In [ ]:
dataset = load_dataset("squad")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained( "distilbert-base-uncased")

model = AutoModelForQuestionAnswering.from_pretrained( "distilbert-base-uncased")


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
qa_outputs.weight       | MISSING    | 
qa_outputs.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:

def preprocess_function(examples):

    questions = [q.strip() for q in examples["question"]]

    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs["offset_mapping"]
    sample_map = inputs["overflow_to_sample_mapping"]

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):

        sample_idx = sample_map[i]

        answers = examples["answers"][sample_idx]

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        sequence_ids = inputs.sequence_ids(i)

        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1

        context_start = idx

        while sequence_ids[idx] == 1:
            idx += 1

        context_end = idx - 1

        if (
            offsets[context_start][0] > end_char
            or offsets[context_end][1] < start_char
        ):

            start_positions.append(0)
            end_positions.append(0)

        else:

            idx = context_start

            while (
                idx <= context_end
                and offsets[idx][0] <= start_char
            ):
                idx += 1

            start_positions.append(idx - 1)

            idx = context_end

            while (
                idx >= context_start
                and offsets[idx][1] >= end_char
            ):
                idx -= 1

            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions

    return inputs

In [ ]:
tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

tokenized_datasets

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'overflow_to_sample_mapping', 'start_positions', 'end_positions'],
        num_rows: 88524
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'overflow_to_sample_mapping', 'start_positions', 'end_positions'],
        num_rows: 10784
    })
})

In [ ]:
data_collator = DefaultDataCollator()

In [ ]:
def compute_metrics(eval_pred):

    start_logits, end_logits = eval_pred.predictions

    start_labels, end_labels = eval_pred.label_ids

    start_preds = np.argmax(start_logits, axis=1)
    end_preds = np.argmax(end_logits, axis=1)

    start_accuracy = accuracy_score(start_labels, start_preds)
    end_accuracy = accuracy_score(end_labels, end_preds)

    avg_accuracy = (start_accuracy + end_accuracy) / 2

    return {
        "accuracy": avg_accuracy
    }

In [ ]:

experiments = [

    {
        "learning_rate": 2e-5,
        "batch_size": 8,
        "epochs": 1
    },

    {
        "learning_rate": 5e-5,
        "batch_size": 8,
        "epochs": 2
    },

    {
        "learning_rate": 3e-5,
        "batch_size": 16,
        "epochs": 2
    },

    {
        "learning_rate": 5e-5,
        "batch_size": 16,
        "epochs": 3
    }
]

In [ ]:
results_list = []


In [ ]:
for idx, exp in enumerate(experiments):

    print("\n")
    print(f"EXPERIMENT {idx+1}")

    print(exp)

    model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-uncased")

    training_args = TrainingArguments(

        output_dir=f"./qa_results_{idx}",

        eval_strategy="epoch",

        save_strategy="epoch",

        learning_rate=exp["learning_rate"],

        per_device_train_batch_size=exp["batch_size"],

        per_device_eval_batch_size=exp["batch_size"],

        num_train_epochs=exp["epochs"],

        weight_decay=0.01,

        logging_steps=100,

        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(

        model=model,

        args=training_args,

        train_dataset=tokenized_datasets["train"].shuffle(seed=42).select(range(5000)),

        eval_dataset=tokenized_datasets["validation"].shuffle(seed=42).select(range(1000)),

        data_collator=data_collator,

        compute_metrics=compute_metrics,
    )

    trainer.train()

    results = trainer.evaluate()

    results_list.append({

        "Learning Rate": exp["learning_rate"],

        "Batch Size": exp["batch_size"],

        "Epochs": exp["epochs"],

        "Accuracy": round(
            results["eval_accuracy"],
            4
        )
    })




EXPERIMENT 1
{'learning_rate': 2e-05, 'batch_size': 8, 'epochs': 1}


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
qa_outputs.weight       | MISSING    | 
qa_outputs.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,2.572173,2.401842,0.385500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss,Accuracy
1,2.057972,1.831990,0.507500
2,1.343240,1.609200,0.557500
3,0.811973,1.767565,0.564000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
0.811973,1.767565,3,0.564000


In [ ]:
results_df = pd.DataFrame(results_list)

print("FINAL HYPERPARAMETER RESULTS")

results_df

FINAL HYPERPARAMETER RESULTS


,Learning Rate,Batch Size,Epochs,Accuracy
0,0.00002,8,1,0.3855
1,0.00005,8,2,0.5790
2,0.00003,16,2,0.5315
3,0.00005,16,3,0.5640
